# Systematics: Flux, G4, MCStat, Cosmics

In [ ]:
%load_ext autoreload
%autoreload 

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

# local imports
import sys
sys.path.append('../../')
from pyanalib.split_df_helpers import *
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.utils import *
from pyanalib.covariance import *
from makedf.mcstat import get_MCstat_unc

import matplotlib.pyplot as plt 
plt.style.use("presentation.mplstyle")

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = False
save_fig = save_result

save_fig_base_dir = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi"
today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, "systematics-other-{}".format(today_str))

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

In [ ]:
file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09"
n_max_concat = 3
mc_keys2load = ['hdr', 'evt'] 

concat_dfs = load_and_concat_mc_dfs(
    file_dir=file_dir,
    chunk_tags=generate_tags("ad"),
    df_tag="",
    keys2load=mc_keys2load,
    n_max_concat=n_max_concat,
    sub_dir="MC",
    sample_dir="BNB_cosmics"
)

mc_hdr_df = concat_dfs['hdr']
mc_evt_df = concat_dfs['evt']

In [ ]:
print("==== breakdown of selected events ====")
mc_evt_df.loc[:,'nuint_categ'] = get_int_category(mc_evt_df)
mc_evt_df.loc[:,'genie_categ'] = get_genie_category(mc_evt_df)
print(mc_evt_df.nuint_categ.value_counts())
print(mc_evt_df.genie_categ.value_counts())

In [ ]:
var_config = VariableConfig.muon_momentum()

# MC Stat

In [ ]:
syst_name = "MCstat"

univ_events, cv_events = get_univ_rates(evtdf=mc_evt_df,
                                        var_config=var_config,
                                        n_univ=1000,
                                        bkgd_subtract=True,
                                        syst_name=syst_name)
ret = get_covariance_matrix(univ_events, cv_events)

In [ ]:
plot_univ_hists(univ_events, cv_events, syst_name, var_config)

frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
plot_frac_unc(frac_unc, var_config)

In [ ]:
matrix_type = "cov"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret[matrix_type], var_config, 
             title=title, 
             save_fig=save_fig, save_fig_name=save_fig_name)

In [ ]:
if 'syst_dict' in locals():
    syst_dict[var_config.var_save_name] = ret["cov_frac"]

else:
    syst_dict = {}
    syst_dict[var_config.var_save_name] = ret["cov_frac"]

# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/mcstat_syst_dict.npz", **syst_dict)

# Flux

In [ ]:
syst_name = ("mc", "Flux")

univ_events, cv_events = get_univ_rates(evtdf=mc_evt_df,
                                        var_config=var_config,
                                        n_univ=100,
                                        bkgd_subtract=True,
                                        syst_name=syst_name)
ret = get_covariance_matrix(univ_events, cv_events)

In [ ]:
plot_univ_hists(univ_events, cv_events, syst_name, var_config)

frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
plot_frac_unc(frac_unc, var_config)

In [ ]:
matrix_type = "cov"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret[matrix_type], var_config, 
             title=title, 
             save_fig=save_fig, save_fig_name=save_fig_name)

In [ ]:
if 'syst_dict' in locals():
    syst_dict[var_config.var_save_name] = ret["cov_frac"]

else:
    syst_dict = {}
    syst_dict[var_config.var_save_name] = ret["cov_frac"]

# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/flux_syst_dict.npz", **syst_dict)

# G4

In [ ]:
syst_name = ("mc", "reinteractions_proton_Geant4")
# syst_name = ("mc", "G4")

univ_events, cv_events = get_univ_rates(evtdf=mc_evt_df,
                                        var_config=var_config,
                                        n_univ=100,
                                        bkgd_subtract=True,
                                        syst_name=syst_name)
ret = get_covariance_matrix(univ_events, cv_events)

In [ ]:
plot_univ_hists(univ_events, cv_events, syst_name, var_config)

frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
plot_frac_unc(frac_unc, var_config)

In [ ]:
matrix_type = "cov"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret[matrix_type], var_config, 
             title=title, 
             save_fig=save_fig, save_fig_name=save_fig_name)

In [ ]:
if 'syst_dict' in locals():
    syst_dict[var_config.var_save_name] = ret["cov_frac"]

else:
    syst_dict = {}
    syst_dict[var_config.var_save_name] = ret["cov_frac"]

# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/g4_syst_dict.npz", **syst_dict)

# Cosmics

take the difference between the offbeam data and intime MC as a unisim uncertainty

In [ ]:
n_max_concat = 10
data_keys2load = ['evt', 'hdr']
data_file = path.join(file_dir, "data", "OffBeam", "aa.df")
data_dfs = load_dfs(data_file, data_keys2load, n_max_concat=n_max_concat)
data_evt_df = data_dfs['evt']
data_hdr_df = data_dfs['hdr']
data_gates = data_hdr_df[data_hdr_df['first_in_subrun'] == 1]['noffbeambnb'].sum()
print("intime cosmics data gates: {:.2e}".format(data_gates))

mc_keys2load = ['evt', 'hdr']
mc_file = path.join(file_dir, "MC", "intime", "aa.df")
mc_dfs = load_dfs(mc_file, mc_keys2load, n_max_concat=n_max_concat)
mc_evt_df = mc_dfs['evt']
mc_hdr_df = mc_dfs['hdr']
mc_gates = mc_hdr_df[mc_hdr_df['first_in_subrun'] == 1]['ngenevt'].sum()
print("intime cosmics MC gates: {:.2e}".format(mc_gates))

scale = data_gates/mc_gates
print("offbeam scale: {:.2f}".format(scale))
mc_evt_df["pot_scale"] = scale

In [ ]:
# compare data and MC
fig, ax = plt.subplots()

data_events, _, _ = plt.hist(data_evt_df[var_config.var_evt_reco_col], bins=var_config.bins, histtype="step", color="red", label="Data")
mc_events, _, _   = plt.hist(mc_evt_df[var_config.var_evt_reco_col], weights=mc_evt_df["pot_scale"], bins=var_config.bins, histtype="step", color="black", label="MC")

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
plt.ylabel("Events / Bin")
plt.legend()
plt.show();

In [ ]:
# treat data as a unisim systematic universe
# the MC-data difference is the variation in event rate

syst_name = "cosmics"
univ_events =np.array([cv_events + (data_events - mc_events)])

ret = get_covariance_matrix(univ_events, cv_events)

In [ ]:
plot_univ_hists(univ_events, cv_events, syst_name, var_config)

frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
plot_frac_unc(frac_unc, var_config)

In [ ]:
matrix_type = "cov"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret[matrix_type], var_config, 
             title=title, 
             save_fig=save_fig, save_fig_name=save_fig_name)

In [ ]:
if 'syst_dict' in locals():
    syst_dict[var_config.var_save_name] = ret["cov_frac"]

else:
    syst_dict = {}
    syst_dict[var_config.var_save_name] = ret["cov_frac"]

# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/cosmics_syst_dict.npz", **syst_dict)

# GENIE

In [ ]:
syst_name = ("mc", "GENIE")

In [ ]:
file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09"
n_max_concat = 3
mc_keys2load = ['hdr', 'mcnu', 'evt'] 

concat_dfs = load_and_concat_mc_dfs(
    file_dir=file_dir,
    chunk_tags=generate_tags("ac"),
    df_tag="_sel_mup-geniewgts",
    keys2load=mc_keys2load,
    n_max_concat=n_max_concat,
    sub_dir="MC",
    sample_dir="BNB_cosmics"
)

mc_hdr_df = concat_dfs['hdr']
mc_nu_df = concat_dfs['mcnu']
mc_evt_df = concat_dfs['evt']

In [ ]:
## total pot
mc_tot_pot = mc_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))

# target_pot = 1e20
target_pot = mc_tot_pot
mc_pot_scale = target_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))

mc_evt_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_evt_df))
mc_nu_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_nu_df))

In [ ]:
print("==== breakdown of selected events ====")
mc_evt_df.loc[:,'nuint_categ'] = get_int_category(mc_evt_df)
mc_evt_df.loc[:,'genie_categ'] = get_genie_category(mc_evt_df)
print(mc_evt_df.nuint_categ.value_counts())
print(mc_evt_df.genie_categ.value_counts())

print("==== breakdown of all MC events ====")
mc_nu_df.loc[:,'nuint_categ'] = get_int_category(mc_nu_df)
mc_nu_df.loc[:,'genie_categ'] = get_genie_category(mc_nu_df)
print(mc_nu_df.nuint_categ.value_counts())
print(mc_nu_df.genie_categ.value_counts())

In [ ]:
signal_hists(evtdf=mc_evt_df, 
             nudf=mc_nu_df, 
             var_config=var_config, 
             save_fig=save_fig, 
             save_name=None)

In [ ]:
# Note: these are background subtracted
cov_type = "xsec"
univ_events, cv_events = get_univ_rates(cov_type, mc_evt_df, mc_nu_df, var_config, syst_name)
ret_xsec = get_covariance_matrix(univ_events, cv_events)
plot_univ_hists(univ_events, cv_events, syst_name, var_config)

cov_type = "rate"
univ_events, cv_events = get_univ_rates(cov_type, mc_evt_df, mc_nu_df, var_config, syst_name)
ret_rate = get_covariance_matrix(univ_events, cv_events)
plot_univ_hists(univ_events, cv_events, syst_name, var_config)

In [ ]:
matrix_type = "cov"
save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
title = "{} {}".format(syst_name, matrix_type)
plot_heatmap(ret_xsec[matrix_type], var_config, 
             title=title, 
             save_fig=save_fig, save_fig_name=save_fig_name)
plot_heatmap(ret_rate[matrix_type], var_config, 
             title=title, 
             save_fig=save_fig, save_fig_name=save_fig_name)

In [ ]:
frac_unc = np.sqrt(np.diag(ret_xsec["cov_frac"]))
plot_frac_unc(frac_unc, var_config)

frac_unc = np.sqrt(np.diag(ret_rate["cov_frac"]))
plot_frac_unc(frac_unc, var_config)

In [ ]:
if 'syst_dict' in locals():
    syst_dict[var_config.var_save_name] = ret_xsec["cov_frac"]
    syst_dict[var_config.var_save_name+"_rate"] = ret_rate["cov_frac"]

else:
    syst_dict = {}
    syst_dict[var_config.var_save_name] = ret_xsec["cov_frac"]
    syst_dict[var_config.var_save_name+"_rate"] = ret_rate["cov_frac"]

# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/genie_syst_dict.npz", **syst_dict)